# Build A MOM6-Solo ISOMIP Case

Use this notebook to configure an ISOMIP-style MOM6-solo case from a GFZ/PIK `Ocean1`-`Ocean4` geometry file, preview the shelf and bed geometry, then generate an isolated payu control that can be run with `payu setup` and `payu run`.

V1 uses a fixed horizontal MOM mask and sets `DYNAMIC_CAVITY_GEOMETRY=False`, matching the validated `ocean3-geometry-noop smoke` path.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception as exc:
    HAS_PLOTLY = False
    print(f"Plotly unavailable: {exc}")

REPO = Path('/g/data/au88/jr5971/isomip-test-mom6-for-iom3-configs')
sys.path.insert(0, str(REPO / 'tools'))

from isomip_case_builder import (
    CaseConfig,
    GeometryTransform,
    GridConfig,
    RuntimeConfig,
    build_generated_case,
    build_geometry,
    geometry_summary,
    payu_commands,
)


## Configuration

Edit these values, then re-run the notebook. The defaults are intentionally close to the validated Ocean3 no-op geometry smoke case.

In [ ]:
CASE_NAME = 'ocean3-generated-geometry-noop-smoke'
OCEAN_CASE = 'Ocean3'          # Ocean1, Ocean2, Ocean3, Ocean4
GEOMETRY_MODE = 'geometry-noop' # static, geometry-noop, geometry

grid = GridConfig(
    nx=240,
    ny=40,
    dx_m=2000.0,
    dy_m=2000.0,
    west_m=320000.0,
    south_m=0.0,
    method='auto',
)

transform = GeometryTransform(
    thickness_scale=1.0,
    front_shift_m=0.0,
    channel_width_m=None,
    min_thickness_m=10.0,
    floating_fraction_threshold=0.0,
    bed_offset_m=0.0,
    bed_slope_m_per_km=0.0,
    min_cavity_depth_m=1.0,
    clear_thin_cavity=True,
)

runtime = RuntimeConfig(
    profile='smoke',
    dt_s=300,
    nk=36,
    regridding_mode='SIGMA_SHELF_ZSTAR',
    queue='normalsr',
    walltime='01:00:00',
    ncpus=48,
    jobname='mom6_iso_gen',
)

config = CaseConfig(
    name=CASE_NAME,
    ocean_case=OCEAN_CASE,
    geometry_mode=GEOMETRY_MODE,
    grid=grid,
    transform=transform,
    runtime=runtime,
    replace=True,
)

data = build_geometry(config)
pd.DataFrame([geometry_summary(data)])


## 2-D Geometry Preview

In [ ]:
time_index = 0
x_km = data.x / 1000.0
y_km = data.y / 1000.0
thick = data.thick[time_index]
base = data.base[time_index]
bed = data.bed[time_index]
frac = data.floating_fraction[time_index]
cavity = np.where(frac > 0, base - bed, np.nan)

fields = [
    ('bed elevation [m]', bed, 'terrain'),
    ('ice base [m]', np.where(frac > 0, base, np.nan), 'Blues_r'),
    ('ice thickness [m]', np.where(frac > 0, thick, np.nan), 'viridis'),
    ('floating fraction', frac, 'magma'),
    ('cavity thickness [m]', cavity, 'cividis'),
    ('active shelf mask', frac > 0, 'gray_r'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 7), constrained_layout=True)
for ax, (title, field, cmap) in zip(axes.ravel(), fields):
    im = ax.pcolormesh(x_km, y_km, field, shading='auto', cmap=cmap)
    ax.set_title(title)
    ax.set_xlabel('x [km]')
    ax.set_ylabel('y [km]')
    ax.set_aspect('equal')
    fig.colorbar(im, ax=ax, shrink=0.85)
plt.show()

if data.warnings:
    print('Warnings:')
    for warning in data.warnings:
        print('-', warning)


## 3-D Geometry Preview

The 3-D plot shows bed, ice base, and ice surface. Use vertical exaggeration carefully; it is for inspection, not measurement.

In [ ]:
vertical_exaggeration = 20.0
surface = np.where(frac > 0, base + thick, np.nan)
base_plot = np.where(frac > 0, base, np.nan)

if HAS_PLOTLY:
    fig = go.Figure()
    fig.add_trace(go.Surface(x=x_km, y=y_km, z=bed * vertical_exaggeration, colorscale='Earth', opacity=0.9, name='bed'))
    fig.add_trace(go.Surface(x=x_km, y=y_km, z=base_plot * vertical_exaggeration, colorscale='Blues', opacity=0.7, name='ice base'))
    fig.add_trace(go.Surface(x=x_km, y=y_km, z=surface * vertical_exaggeration, colorscale='Ice', opacity=0.55, name='ice surface'))
    fig.update_layout(
        title=f'{OCEAN_CASE} generated geometry, t={time_index}',
        scene=dict(xaxis_title='x [km]', yaxis_title='y [km]', zaxis_title=f'z * {vertical_exaggeration:g} [m]'),
        height=700,
    )
    fig.show()
else:
    print('Install/enable plotly for the interactive 3-D view.')


## Generate The Payu Control

Set `BUILD_CONTROL = True` when you are happy with the geometry. The notebook creates a fresh isolated control and lab, updates manifests, and prints the exact commands to run.

In [ ]:
BUILD_CONTROL = False

if BUILD_CONTROL:
    paths = build_generated_case(config)
    print(f'control: {paths.control}')
    print(f'lab:     {paths.lab}')
    print(f'summary: {paths.summary_json}')
    print('\nRun commands:')
    for command in payu_commands(paths.control, paths.lab):
        print(command)
else:
    control = config.control_root / config.name
    lab = config.lab_root / config.name
    print('Preview only. Set BUILD_CONTROL = True to write the control.')
    print('\nPlanned run commands:')
    for command in payu_commands(control, lab):
        print(command)
